# 第 7 周：QLoRA 微调 —— 在特定任务上挑战前沿模型

## 练习目标（理念）

本笔记本展示 **QLoRA** 的理解与实现：在**特定任务**（由医患对话生成结构化临床记录）上微调**开源模型**，并与前沿模型（经 OpenRouter 的 GPT-4）对比，看能否在该窄任务上接近或超过通才大模型。

- **任务：** 输入对话 → 输出结构化临床笔记（范围窄、定义清晰）。
- **QLoRA：** 4-bit 量化基座（NF4 + 双重量化）+ LoRA 适配器，微调时显存可控。
- **LoRA：** 低秩适配（如 `q_proj`、`v_proj`）；只训练约 0.2% 参数。
- **对比前沿：** 同一评估子集上比较微调开源模型 vs GPT-4；任务数据够专时，小模型也可匹配通才模型。

## 和本课 Week 7 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 4-bit 量化 | `BitsAndBytesConfig` + NF4 |
| LoRA / QLoRA | `LoraConfig` + `prepare_model_for_kbit_training` |
| 监督微调 SFT | `SFTTrainer` / `SFTConfig`，`dataset_text_field="text"` |
| 任务专用数据 | ACI-Bench 医患对话 → 临床笔记 |
| 与前沿对比 | OpenRouter 调 `openai/gpt-4`（可选） |

**默认模型：** TinyLlama（通常无需申请）。可选：[LLaMA 3.2](https://huggingface.co/meta-llama/Llama-3.2-3B)——需设置 `HF_TOKEN` 并改 `MODEL_NAME`。

## 怎么跑

1. **按顺序：** 先跑安装格（单元 1）一次，再 **Kernel → Run All**，保证变量都已定义。
2. **模型：** 默认 TinyLlama。换 LLaMA 3.2：先获访问权限，设环境变量 `HF_TOKEN`，并改配置格里的 `MODEL_NAME`。
3. **Weights & Biases：** 可选；训练可不依赖它。若要用，把 `report_to` 设为 `"wandb"` 并运行 `wandb.login()`。
4. **OpenRouter（GPT-4 对比）：** 可选；仅比较时需要环境变量 `OPENROUTER_API_KEY`。
5. **GPU：** 训练需要 GPU（如 Colab T4/A100）。纯 CPU 会极慢或 OOM。


In [ ]:
# ========== 安装依赖（本格跑一次即可；若 peft 仍缺失请重启内核后再 Run All）==========
# subprocess 调当前解释器的 pip，避免装到别的 Python 环境
import subprocess, sys
# -q 安静安装：训练/推理/量化/数据集/实验追踪/演示 UI 所需包
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "torch", "transformers", "accelerate", "peft", "trl", "bitsandbytes", "datasets", "wandb", "gradio", "requests"])


In [ ]:
# ========== 导入：QLoRA 训练、评估与 Gradio 演示 ==========

# PyTorch：张量与设备（CUDA / MPS）
import torch
# transformers 生态（版本信息等）
import transformers
# 分词器、因果 LM、BitsAndBytes 量化配置
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# LoRA 配置、挂载适配器、k-bit 训练准备
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# TRL 的 SFT 训练参数与 Trainer
from trl import SFTConfig, SFTTrainer
# 从 Hub 加载医患对话数据集
from datasets import load_dataset
# Weights & Biases：可选实验追踪
import wandb
# json：结构化数据（本笔记本主要靠 datasets，保留原导入）
import json
# os：读 HF_TOKEN / OPENROUTER_API_KEY 等环境变量
import os
# numpy：数值工具
import numpy as np
# tqdm：评估循环进度条
from tqdm import tqdm
# requests：调用 OpenRouter HTTP API
import requests
# Gradio：浏览器里试临床笔记生成
import gradio as gr


In [ ]:
# ========== QLoRA + LoRA 超参与路径（开源模型 × 特定任务 × 对比前沿）==========

# 默认小聊天模型；可换成 meta-llama/Llama-3.2-3B（需 HF_TOKEN）
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # or "meta-llama/Llama-3.2-3B" + HF_TOKEN
# Hugging Face 令牌（门禁模型 / 部分数据集可能需要）
HF_TOKEN = os.getenv("HF_TOKEN")

# 任务数据：对话 → 临床笔记；输出目录与 W&B 项目名
DATASET_NAME = "ClinicianFOCUS/ACI-Bench-Refined"
OUTPUT_DIR = "./lora-medical-llama"
WANDB_PROJECT = "qlora-medical-extraction"

# LoRA：秩 r、缩放 alpha、dropout、只改注意力 q/v 投影
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj"]

# 训练超参：学习率、batch、梯度累积、轮数、序列长、日志/评估/保存步频
LEARNING_RATE = 2e-4
BATCH_SIZE = 4
GRAD_ACC_STEPS = 4
EPOCHS = 3
MAX_SEQ_LENGTH = 512
WARMUP_STEPS = 100
LOGGING_STEPS = 10
EVAL_STEPS = 200
SAVE_STEPS = 200

# QLoRA：4-bit NF4 + 双重量化，让基座塞进较小显存
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# OpenRouter API Key（可选；后面和 GPT-4 对比时才需要）
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")


In [ ]:
# ========== 加载数据并格式化为 SFT 的 text 字段 ==========

# 按 DATASET_NAME 拉取；token 传 HF_TOKEN（无私有权限时也可为 None）
dataset = load_dataset(DATASET_NAME, token=HF_TOKEN)
train_dataset = dataset["train"]

# 从 train 再切 90/10 验证集（固定 seed=42 可复现）
split_dataset = train_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# 把一条样本拼成 Instruction / Response 文本（prompt 原文保持英文，供模型学习）
def format_instruction(example):
    instruction = f"Generate a structured clinical note from the following doctor-patient dialogue:\n{example['dialogue']}"
    response = example['note']
    return f"### Instruction:\n{instruction}\n\n### Response:\n{response}"

train_dataset = train_dataset.map(lambda x: {"text": format_instruction(x)})
eval_dataset = eval_dataset.map(lambda x: {"text": format_instruction(x)})

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")


In [ ]:
# ========== 分词器：与 MODEL_NAME 对齐 ==========

# 从 Hub 加载与基座同名的 tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# 无 pad_token 时用 eos 顶上，满足 Trainer / 批量 padding
tokenizer.pad_token = tokenizer.eos_token


In [ ]:
# ========== 按设备选择 QLoRA（CUDA）或纯 LoRA+bf16（MPS/CPU）==========

# Apple Silicon 上 bitsandbytes 4bit/8bit 优化器常不可用 → bf16 + LoRA
_use_cuda = torch.cuda.is_available()
_is_mps = getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available()
_use_qlora = _use_cuda  # QLoRA only on CUDA; on MPS use full bf16 + LoRA so training runs

if _use_qlora:
    # CUDA：4-bit 基座 + 后续 LoRA（真正的 QLoRA 路径）
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN,
    )
    # 冻结量化基座并打开梯度检查点等 k-bit 训练准备
    model = prepare_model_for_kbit_training(model)
else:
    # MPS/CPU：整模 bf16 + LoRA，避开 bitsandbytes 限制
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN,
    )

# LoRA 超参：只训 target_modules，bias 不训，任务类型为因果 LM
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)
# 把 LoRA 挂到模型上
model = get_peft_model(model, lora_config)
# 打印可训练参数占比（应远小于 100%）
model.print_trainable_parameters()
if _is_mps:
    print("(MPS: using bf16 + LoRA; run on Colab/CUDA for QLoRA.)")


In [ ]:
# ========== SFTConfig：训练参数 + dataset_text_field / max_length ==========

# CUDA 可用 8bit paged AdamW；MPS/CPU 必须 adamw_torch
_optim = "paged_adamw_8bit" if _use_cuda else "adamw_torch"
# pin_memory 仅在 CUDA DataLoader 上有意义
_dataloader_pin_memory = _use_cuda

# TRL SFTConfig：普通 TrainingArguments 字段 + SFT 专用字段
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=EPOCHS,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    optim=_optim,
    dataloader_pin_memory=_dataloader_pin_memory,
    report_to="none",
    run_name="medical-llama-qlora",
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
)


In [ ]:
# 可选：若把上面 report_to 改成 "wandb"，取消下一行注释并粘贴 API Key
# wandb.login()


In [ ]:
# ========== 组装 SFTTrainer：模型 + 参数 + 训练/验证集 + 分词器 ==========

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    # TRL 新接口用 processing_class 传入 tokenizer
    processing_class=tokenizer,
)


In [ ]:
# ========== 开始训练并落盘适配器 / 分词器 ==========

# 监督微调主循环（耗时；需 GPU）
trainer.train()
# 保存 LoRA（及配置）到 OUTPUT_DIR
trainer.save_model(OUTPUT_DIR)
# 分词器一并保存，便于推理时 from_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
# wandb.finish() # 仅当您使用 report_to="wandb" 时


In [ ]:
# ========== 简易验证：生成临床笔记并做启发式「准确率」==========

def evaluate_model(model, tokenizer, eval_dataset, num_samples=50):
    # 评估模式：关闭 dropout 等
    model.eval()
    correct = 0
    total = 0
    # 最多取 num_samples 条，避免全量生成太慢
    for example in tqdm(eval_dataset.select(range(min(num_samples, len(eval_dataset))))):
        # 只保留 Instruction 段，截到 Response 提示符，让模型续写
        prompt = example["text"].split("### Response:\n")[0] + "### Response:\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # 抽出模型生成的 Response 正文
        if "### Response:\n" in response:
            pred = response.split("### Response:\n")[-1].strip()
        else:
            pred = ""
        # 参考答案（金标笔记）
        ref = example["text"].split("### Response:\n")[-1].strip()
        # 简单的启发式：检查是否像注释一样（可以改进）
        if "note" in pred.lower() and len(pred) > 50:
            correct += 1
        total += 1
    return correct / total if total > 0 else 0

accuracy = evaluate_model(model, tokenizer, eval_dataset)
print(f"Validation accuracy: {accuracy:.4f}")


In [ ]:
# ========== 可选：用 OpenRouter 调 GPT-4，与微调开源模型对比 ==========

def query_openrouter(prompt, model="openai/gpt-4"):
    # POST Chat Completions；Authorization 用环境变量里的 Key
    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.1,
            "max_tokens": 512,
        }
    )
    return response.json()["choices"][0]["message"]["content"]

subset_size = 20
eval_subset = eval_dataset.select(range(min(subset_size, len(eval_dataset))))
gpt4_accuracy = None
if OPENROUTER_API_KEY:
    gpt4_correct = 0
    for example in tqdm(eval_subset):
        # 与微调模型相同的 prompt 截取方式
        prompt = example["text"].split("### Response:\n")[0] + "### Response:\n"
        try:
            gpt4_pred = query_openrouter(prompt, model="openai/gpt-4")
        except Exception:
            gpt4_pred = ""
        # 同一套启发式打分，便于粗对比
        if "note" in gpt4_pred.lower() and len(gpt4_pred) > 50:
            gpt4_correct += 1
    gpt4_accuracy = gpt4_correct / subset_size
    print(f"GPT-4 accuracy: {gpt4_accuracy:.4f}")
print(f"Your QLoRA (fine-tuned open-source) accuracy: {accuracy:.4f}")
if gpt4_accuracy is not None:
    print(f"→ Outperform frontier? {'Yes' if accuracy >= gpt4_accuracy else 'Close / No'} (vs GPT-4 {gpt4_accuracy:.4f})")


In [ ]:
# 从 peft 再导入 PeftModel：推理时「基座 + 已保存 LoRA」
from peft import PeftModel

# 重新加载基础模型 + LoRA 权重进行推理
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    token=HF_TOKEN,
)
# 分词器从 OUTPUT_DIR 读（与训练结束时保存的一致）
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
# 把磁盘上的 LoRA 挂到刚加载的基座
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()

# Gradio 回调：对话 → 生成结构化临床笔记
def generate_response(dialogue, temperature=0.1, max_new_tokens=256):
    # Instruction 模板须与训练 format_instruction 一致（英文 prompt 不改）
    prompt = f"### Instruction:\nGenerate a structured clinical note from the following doctor-patient dialogue:\n{dialogue}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 只返回 Response 之后的正文
    if "### Response:\n" in response:
        response = response.split("### Response:\n")[-1].strip()
    return response

iface = gr.Interface(
    fn=generate_response,
    inputs=[
        gr.Textbox(lines=5, placeholder="Enter doctor-patient dialogue...", label="Dialogue"),
        gr.Slider(0.0, 1.0, value=0.1, label="Temperature"),
        gr.Slider(64, 512, value=256, step=32, label="Max New Tokens")
    ],
    outputs=gr.Textbox(lines=10, label="Generated Clinical Note"),
    title="Medical Note Generator (QLoRA fine-tuned open-source model)",
    description="Enter a dialogue between doctor and patient to generate a structured clinical note."
)

# 无头/无显示：share=True 给出公共 URL； inbrowser=False 跳过打开浏览器
iface.launch(share=True, inbrowser=False)
# 在任何设备上的任何浏览器中打开打印的 URL（例如 https://xxx.gradio.live）。
